# ResearchGPT V2 ? private T4 inference, contract fix v3

Authoring status: **NOT EXECUTED on Kaggle**. The previous job was reported to finish generation but fail schema validation. This is a separate corrected package, not a repaired model response. No remote notebook URL or ID has been supplied.

Upload packet.json, request.json, runtime.json, code.zip and input-manifest.json from the v3 export as one **private dataset**, attach it to this **private notebook**, select **Settings > Accelerator > GPU T4 x2** when available, and enable Internet for the pinned public dependencies/model. The bootstrap discovers the actual mounted path and verifies the retained local manifest SHA. It uses physical GPU 0 only.

The runner constrains generation to the exact source-ID selection schema and retains independent post-validation. The trusted local host still verifies evidence. See kaggle/INFERENCE.md and docs/QWEN_CONTRACT_FIX.md. Execute cells in order once; preserve any failure without truncating candidates, inventing IDs, or changing the evidence.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import zipfile

# Retained from the trusted local export, not computed from an untrusted upload.
rgpt_expected_sha = 'f4364f63743f927be904025b3c404045a3fa56a2e89e14e6fa408bbd7e72980f'
rgpt_input_root = Path('/kaggle/input')
rgpt_work_root = Path('/kaggle/working')
rgpt_physical_device = 0


In [ ]:
rgpt_matches = [p for p in rgpt_input_root.rglob('input-manifest.json')
                if hashlib.sha256(p.read_bytes()).hexdigest() == rgpt_expected_sha]
if len(rgpt_matches) != 1:
    raise RuntimeError('Attach exactly one private export matching the retained local manifest SHA; found ' + str(len(rgpt_matches)))
rgpt_input = rgpt_matches[0].parent
rgpt_manifest = json.loads(rgpt_matches[0].read_text(encoding='utf-8'))
if set(rgpt_manifest['files']) != {'packet.json', 'request.json', 'runtime.json', 'code.zip'}:
    raise ValueError('Unexpected input files')
for rgpt_name, rgpt_entry in rgpt_manifest['files'].items():
    rgpt_content = (rgpt_input / rgpt_name).read_bytes()
    if len(rgpt_content) != rgpt_entry['bytes'] or hashlib.sha256(rgpt_content).hexdigest() != rgpt_entry['sha256']:
        raise ValueError('Input integrity mismatch: ' + rgpt_name)
rgpt_code = rgpt_work_root / ('rgpt-source-' + rgpt_manifest['code_commit'][:7])
if rgpt_code.exists():
    raise FileExistsError('Source directory already exists; use a fresh session/work directory, preserving earlier results')
with zipfile.ZipFile(rgpt_input / 'code.zip') as rgpt_archive:
    for rgpt_member in rgpt_archive.infolist():
        rgpt_path = Path(rgpt_member.filename)
        if rgpt_path.is_absolute() or '..' in rgpt_path.parts or '\\' in rgpt_member.filename:
            raise ValueError('Unsafe code archive member')
    rgpt_archive.extractall(rgpt_code)
print(json.dumps({'package_id': rgpt_manifest['package_id'],
                  'input_manifest_sha256': rgpt_expected_sha,
                  'actual_mounted_input': str(rgpt_input),
                  'code_commit': rgpt_manifest['code_commit']}, indent=2))


## Install in a separate environment

These prepared pins have not been installation-tested on a remote Kaggle session. This cell creates a new environment and leaves the notebook's base Python untouched. A nonzero command stops setup. No local Windows environment is modified.


In [ ]:
rgpt_env = rgpt_work_root / 'rgpt-inference-env'
if rgpt_env.exists():
    raise FileExistsError('Use a fresh environment; do not overwrite an earlier run')
subprocess.run([sys.executable, '-m', 'venv', str(rgpt_env)], check=True)
rgpt_python = rgpt_env / 'bin' / 'python'
subprocess.run([str(rgpt_python), '-m', 'pip', 'install', 'torch==2.8.0+cu126',
                '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([str(rgpt_python), '-m', 'pip', 'install', '-r',
                str(rgpt_code / 'kaggle' / 'requirements-extraction.txt')], check=True)
subprocess.run([str(rgpt_python), '-m', 'pip', 'check'], check=True)
subprocess.run([str(rgpt_python), '-c', 'import sys; print(sys.executable); print(sys.version)'], check=True)


## Inspect devices and execute one request

The runner masks one physical GPU before importing torch, measures its actual free VRAM before loading, and enforces the exported model's guard. The prepared job requests pinned Qwen3-8B, NF4 double quantization, FP16 compute, non-thinking mode, an 8192-token context and a 2048-token output cap. It preserves the exact messages and rejects truncation or substitution. Actual package/model/configuration data in the export is authoritative.

Run the inference cell once. GPU quota or T4 availability is not guaranteed. If setup or validation fails before the runner starts, retain the notebook error; there may be no result archive. Runner failures after initialization retain `result.zip` where possible.


In [ ]:
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.free,memory.total', '--format=csv'], check=True)
rgpt_runtime = json.loads((rgpt_input / 'runtime.json').read_text(encoding='utf-8'))
print(json.dumps({'physical_device': rgpt_physical_device,
                  'requested_model': rgpt_runtime['requested_model'],
                  'generation': rgpt_runtime['generation'],
                  'quantization': rgpt_runtime['quantization']}, indent=2))


In [ ]:
rgpt_result = rgpt_work_root / 'rgpt-result-01'
if rgpt_result.exists():
    raise FileExistsError('A result directory already exists; preserve it rather than rerunning this request')
rgpt_command = [str(rgpt_python), '-m', 'kaggle.run_inference',
                '--input', str(rgpt_input), '--expected-input-sha256', rgpt_expected_sha,
                '--device', str(rgpt_physical_device), '--output', str(rgpt_result)]
rgpt_child_env = os.environ.copy()
rgpt_child_env['PYTHONPATH'] = str(rgpt_code)
rgpt_run = subprocess.run(rgpt_command, cwd=rgpt_code, env=rgpt_child_env, check=False)
print(json.dumps({'runner_exit_code': rgpt_run.returncode, 'result_directory': str(rgpt_result)}))


## Download and verify locally

Download the generated `result.zip` from the notebook Output browser, including when the recorded execution failed. Keep notebook and outputs private. The result archive contains `raw_response.txt`, `candidates.json`, `execution.json` and `output-manifest.json`. Actual token counts, completion state, timings and memory belong to `execution.json`; do not treat a model's claimed verdict as source verification.

On the local V2 machine, unpack the archive into a fresh result directory and use the original trusted input and retained manifest SHA with `scripts.inference_package import`, as documented in `kaggle/INFERENCE.md`. Stop the GPU session after retaining its output.


In [ ]:
rgpt_result_zip = rgpt_result / 'result.zip'
if not rgpt_result_zip.is_file():
    raise FileNotFoundError('No result archive was produced; retain the setup/runner error and do not report a completed inference')
print(json.dumps({'download': str(rgpt_result_zip),
                  'sha256': hashlib.sha256(rgpt_result_zip.read_bytes()).hexdigest()}, indent=2))
print((rgpt_result / 'execution.json').read_text(encoding='utf-8'))
from IPython.display import FileLink, display
display(FileLink(str(rgpt_result_zip)))
